In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv('all_accelerometer_data_pids_13.csv')

In [3]:
df.head()

,time,pid,x,y,z
0,0,JB3156,0.0000,0.0000,0.0000
1,0,CC6740,0.0000,0.0000,0.0000
2,1493733882409,SA0297,0.0758,0.0273,-0.0102
3,1493733882455,SA0297,-0.0359,0.0794,0.0037
4,1493733882500,SA0297,-0.2427,-0.0861,-0.0163


In [5]:
print(df.shape)

(14057567, 5)


In [6]:
print(df.dtypes)
print(df['pid'].unique())  # pid 이름 확인

time      int64
pid         str
x       float64
y       float64
z       float64
dtype: object
<StringArray>
['JB3156', 'CC6740', 'SA0297', 'PC6771', 'BK7610', 'DC6359', 'MC7070',
 'MJ8002', 'BU4707', 'JR8022', 'HV0618', 'SF3079', 'DK3500']
Length: 13, dtype: str


In [7]:
# TAC 파일 하나 열어서 확인
tac_files = sorted(os.listdir('clean_tac/'))
print(tac_files)  # 파일명 목록

['BK7610_clean_TAC.csv', 'BU4707_clean_TAC.csv', 'CC6740_clean_TAC.csv', 'DC6359_clean_TAC.csv', 'DK3500_clean_TAC.csv', 'HV0618_clean_TAC.csv', 'JB3156_clean_TAC.csv', 'JR8022_clean_TAC.csv', 'MC7070_clean_TAC.csv', 'MJ8002_clean_TAC.csv', 'PC6771_clean_TAC.csv', 'SA0297_clean_TAC.csv', 'SF3079_clean_TAC.csv']


In [ ]:
all_tac_list = []

for filename in tac_files:
    if not filename.endswith('.csv'):
        continue

    pid = filename.split('_')[0]
    temp_df = pd.read_csv(f'clean_tac/{filename}')
    temp_df['pid'] = pid

    # 음수 노이즈 제거
    temp_df['TAC_Reading'] = temp_df['TAC_Reading'].clip(lower=0)

    # 레이블 생성
    temp_df['label'] = (temp_df['TAC_Reading'] >= 0.08).astype(int)

    # TAC_Reading > 0.08, label=1 (drunk), else label=0.

    # timestamp(초) → time(밀리초) 변환
    temp_df['time'] = temp_df['timestamp'] * 1000

    all_tac_list.append(temp_df)

df_tac = pd.concat(all_tac_list, ignore_index=True)

# 확인
print("df_tac 컬럼:", df_tac.columns.tolist())
print("df_tac 행 수:", len(df_tac))
print(df_tac.head(30))

df_tac 컬럼: ['timestamp', 'TAC_Reading', 'pid', 'label', 'time']
df_tac 행 수: 715
     timestamp  TAC_Reading     pid  label           time
0   1493718714     0.000000  BK7610      0  1493718714000
1   1493720697     0.001573  BK7610      0  1493720697000
2   1493721027     0.002144  BK7610      0  1493721027000
3   1493721357     0.000877  BK7610      0  1493721357000
4   1493721686     0.000000  BK7610      0  1493721686000
5   1493722016     0.000000  BK7610      0  1493722016000
6   1493722345     0.000000  BK7610      0  1493722345000
7   1493722674     0.001808  BK7610      0  1493722674000
8   1493723003     0.004542  BK7610      0  1493723003000
9   1493724832     0.005185  BK7610      0  1493724832000
10  1493725160     0.003094  BK7610      0  1493725160000
11  1493725474     0.000000  BK7610      0  1493725474000
12  1493725804     0.000000  BK7610      0  1493725804000
13  1493727636     0.000744  BK7610      0  1493727636000
14  1493729467     0.009188  BK7610      0  149372

## HERE !! ?? thinking about adding 'between' state. w~0.08. we can add increase accuracy..?
---------------------------------------------------------------------------------------

In [11]:
for fname in tac_files:
    sample = pd.read_csv(f'clean_tac/{fname}')
    pid    = fname.split('_')[0]
    print(f"=== {pid} ===")
    print(f"  shape: {sample.shape}")
    print(f"  columns: {sample.columns.tolist()}")
    print(sample.head(3).to_string())
    print()

=== BK7610 ===
  shape: (57, 2)
  columns: ['timestamp', 'TAC_Reading']
    timestamp  TAC_Reading
0  1493718714    -0.000482
1  1493720697     0.001573
2  1493721027     0.002144

=== BU4707 ===
  shape: (57, 2)
  columns: ['timestamp', 'TAC_Reading']
    timestamp  TAC_Reading
0  1493718714    -0.000482
1  1493720697     0.001573
2  1493721027     0.002144

=== CC6740 ===
  shape: (56, 2)
  columns: ['timestamp', 'TAC_Reading']
    timestamp  TAC_Reading
0  1493723434     0.003947
1  1493725257     0.000898
2  1493725585    -0.000894

=== DC6359 ===
  shape: (55, 2)
  columns: ['timestamp', 'TAC_Reading']
    timestamp  TAC_Reading
0  1493719224    -0.002079
1  1493721045     0.000898
2  1493721360     0.002095

=== DK3500 ===
  shape: (51, 2)
  columns: ['timestamp', 'TAC_Reading']
    timestamp  TAC_Reading
0  1493727820     0.000215
1  1493728019     0.001716
2  1493729841     0.001921

=== HV0618 ===
  shape: (54, 2)
  columns: ['timestamp', 'TAC_Reading']
    timestamp  TAC_Read

'''가속도: 40Hz * 28시간 = 약 4,032,000 샘플 (1명당)
TAC:    30분 간격 * 28시간 = 57행 (1명당)

→ 가속도 1행당 TAC 1개를 붙여야 함
→ 30분 구간 안의 모든 가속도에 같은 TAC값이 붙음'''

In [12]:
# TAC 파일 원본 vs clean 비교
pid_check = 'BK7610'
tac = pd.read_csv(f'clean_tac/{pid_check}_clean_TAC.csv')

print(f"=== {pid_check} TAC ===")
print(f"행 수: {len(tac)}")
print(f"컬럼: {tac.columns.tolist()}")
print()
print(tac.to_string())
print("data loaded successfully.")

=== BK7610 TAC ===
행 수: 57
컬럼: ['timestamp', 'TAC_Reading']

     timestamp  TAC_Reading
0   1493718714    -0.000482
1   1493720697     0.001573
2   1493721027     0.002144
3   1493721357     0.000877
4   1493721686    -0.001145
5   1493722016    -0.002159
6   1493722345    -0.001033
7   1493722674     0.001808
8   1493723003     0.004542
9   1493724832     0.005185
10  1493725160     0.003094
11  1493725474    -0.000291
12  1493725804    -0.002025
13  1493727636     0.000744
14  1493729467     0.009188
15  1493731296     0.022250
16  1493733371     0.037422
17  1493735217     0.052238
18  1493737046     0.065357
19  1493738847     0.076462
20  1493740845     0.085197
21  1493742871     0.090165
22  1493744843     0.089122
23  1493746883     0.080588
24  1493748731     0.065953
25  1493750580     0.050424
26  1493752430     0.041689
27  1493754265     0.046559
28  1493756113     0.067269
29  1493757960     0.099630
30  1493759807     0.134315
31  1493761652     0.160774
32  1493763497 

# time matching check
## clean_tac 
## vs 
## all_accelerometer_data_pids_13

In [13]:
# 단위 직접 비교 — datetime 변환 없이
pid_check = 'BK7610'

accel_start = df[df['pid'] == pid_check]['time'].min()
accel_end   = df[df['pid'] == pid_check]['time'].max()
tac_start   = df_tac[df_tac['pid'] == pid_check]['time'].min()
tac_end     = df_tac[df_tac['pid'] == pid_check]['time'].max()

print(f"가속도 time 시작 : {accel_start}")
print(f"가속도 time 끝   : {accel_end}")
print(f"TAC    time 시작 : {tac_start}")
print(f"TAC    time 끝   : {tac_end}")
print()
print(f"자릿수 가속도: {len(str(accel_start))}")
print(f"자릿수 TAC   : {len(str(tac_start))}")
print(f"단위 일치    : {len(str(accel_start)) == len(str(tac_start))}")
print()
print(f"TAC가 가속도 시간 범위 안에 있나: {accel_start <= tac_start <= accel_end}")
print(f"겹치는 구간 (밀리초): {min(accel_end, tac_end) - max(accel_start, tac_start)}")

가속도 time 시작 : 1493735870653
가속도 time 끝   : 1493767770640
TAC    time 시작 : 1493718714000
TAC    time 끝   : 1493807899000

자릿수 가속도: 13
자릿수 TAC   : 13
단위 일치    : True

TAC가 가속도 시간 범위 안에 있나: False
겹치는 구간 (밀리초): 31899987


## BK7610 — Timestamp Alignment Verification

| | Start | End |
|---|---|---|
| Accelerometer | 1493735870653 ms | 1493767770640 ms |
| TAC | 1493718714000 ms | 1493807899000 ms |

**Digit count match:** Both 13 digits (milliseconds) 
**Overlapping window:** 31,899,987 ms ≈ 531 minutes ≈ 8.9 hours 

**Why TAC starts before accelerometer:**
TAC measurement began at ~09:51 UTC to establish a sober baseline,
while accelerometer recording started at ~14:37 UTC
when the bar crawl event began.
This is expected behavior — not a data error.

**Implication for merge_asof (backward):**
All accelerometer rows fall within the TAC recording window.
Without tolerance constraint, every accelerometer row
receives a valid TAC label from the most recent prior TAC reading.

# Run Under line!!

In [14]:
combined_list = []

for pid in sorted(df['pid'].unique()):
    accel_pid = df[df['pid'] == pid].sort_values('time').reset_index(drop=True)
    tac_pid   = df_tac[df_tac['pid'] == pid].sort_values('time').reset_index(drop=True)

    merged_pid = pd.merge_asof(
        accel_pid,
        tac_pid[['time', 'label', 'TAC_Reading']],  # pid 컬럼 제거
        on='time',
        direction='backward'
    )
    # pid 컬럼 직접 추가
    merged_pid['pid'] = pid

    combined_list.append(merged_pid)
    print(f"{pid}: {len(merged_pid):,}rows | "
          f"NaN {merged_pid['label'].isna().sum():,}count")

combined_df = pd.concat(combined_list, ignore_index=True)
combined_df = combined_df.dropna(subset=['label'])
combined_df['label'] = combined_df['label'].astype(int)

print(f"\nall: {len(combined_df):,}rows")
print(f"drunk rate: {combined_df['label'].mean()*100:.1f}%")

print("\n=== pid별 결과 ===")
for pid, group in combined_df.groupby('pid'):
    print(f"{pid}: {len(group):,}rows | drunk {group['label'].mean()*100:.1f}%")

BK7610: 1,225,727rows | NaN 0count
BU4707: 447,423rows | NaN 0count
CC6740: 2,374,695rows | NaN 1count
DC6359: 591,358rows | NaN 0count
DK3500: 1,339,622rows | NaN 0count
HV0618: 1,876,013rows | NaN 0count
JB3156: 1,177,749rows | NaN 1count
JR8022: 307,526rows | NaN 0count
MC7070: 318,600rows | NaN 0count
MJ8002: 631,303rows | NaN 0count
PC6771: 2,141,701rows | NaN 0count
SA0297: 962,901rows | NaN 0count
SF3079: 662,949rows | NaN 0count

all: 14,057,565rows
drunk rate: 23.8%

=== pid별 결과 ===
BK7610: 1,225,727rows | drunk 54.3%
BU4707: 447,423rows | drunk 36.0%
CC6740: 2,374,694rows | drunk 18.6%
DC6359: 591,358rows | drunk 16.7%
DK3500: 1,339,622rows | drunk 0.0%
HV0618: 1,876,013rows | drunk 4.9%
JB3156: 1,177,748rows | drunk 18.4%
JR8022: 307,526rows | drunk 99.5%
MC7070: 318,600rows | drunk 87.7%
MJ8002: 631,303rows | drunk 30.5%
PC6771: 2,141,701rows | drunk 4.3%
SA0297: 962,901rows | drunk 14.7%
SF3079: 662,949rows | drunk 99.9%


# Is person id(pid) : SF3079 merging error? 
### SF3079 : 662,949 row | drunk 99.9%

## EDA — TAC 파일 구조 확인
아래 셀은 데이터 검증 목적으로 작성된 탐색 코드입니다.
최종 파이프라인 실행 시 건너뛰어도 됩니다.

## EDA — TAC file structure test
Under this line until starting 'pipeline' Markshell is written for searching and finding good model trianing & sampling.
when you doing final Running, you can skip this

In [15]:
# SF3079 TAC all rows
sf_tac = df_tac[df_tac['pid'] == 'SF3079'].sort_values('time').reset_index(drop=True)
print(f"SF3079 TAC 행 수: {len(sf_tac)}")
print(sf_tac[['timestamp', 'time', 'TAC_Reading', 'label']].to_string())

SF3079 TAC 행 수: 54
     timestamp           time  TAC_Reading  label
0   1493722490  1493722490000     0.000000      0
1   1493724435  1493724435000     0.000000      0
2   1493724750  1493724750000     0.000000      0
3   1493725064  1493725064000     0.000510      0
4   1493725378  1493725378000     0.015509      0
5   1493725711  1493725711000     0.021869      0
6   1493726043  1493726043000     0.014518      0
7   1493727874  1493727874000     0.000030      0
8   1493728206  1493728206000     0.000000      0
9   1493730037  1493730037000     0.012789      0
10  1493731882  1493731882000     0.059805      0
11  1493733728  1493733728000     0.123564      1
12  1493735575  1493735575000     0.181307      1
13  1493737421  1493737421000     0.211410      1
14  1493739266  1493739266000     0.205234      1
15  1493741116  1493741116000     0.171436      1
16  1493742961  1493742961000     0.130305      1
17  1493744810  1493744810000     0.101965      1
18  1493746658  1493746658000  

In [16]:
# SF3079 가속도 시간 범위
# SF3079 time range for accelerometer
sf_accel = df[df['pid'] == 'SF3079']
print(f"\nAcc Start: {sf_accel['time'].min()}")
print(f"Acc End  : {sf_accel['time'].max()}")
print(f"TAC  Start  : {sf_tac['time'].min()}")
print(f"TAC  End    : {sf_tac['time'].max()}")


Acc Start: 1493739196712
Acc End  : 1493790825495
TAC  Start  : 1493722490000
TAC  End    : 1493805310000


In [17]:
# SF3079 merge 결과 확인 — TAC값이 실제로 어떻게 붙었는지
sf_merged = combined_df[combined_df['pid'] == 'SF3079']
print(f"\nTAC_Reading 분포:")
print(sf_merged['TAC_Reading'].describe())
print(f"\nlabel=1 (drunk) 비율: {sf_merged['label'].mean()*100:.1f}%")

# TAC값 구간별 행 수
bins = [0, 0.02, 0.04, 0.08, 0.15, 0.30, 1.0]
print(sf_merged.groupby(pd.cut(sf_merged['TAC_Reading'], bins))['label'].count())


TAC_Reading 분포:
count    662949.000000
mean          0.128775
std           0.038718
min           0.039047
25%           0.095807
50%           0.130305
75%           0.159026
max           0.211410
Name: TAC_Reading, dtype: float64

label=1 (drunk) 비율: 99.9%
TAC_Reading
(0.02, 0.04]       494
(0.08, 0.15]    474050
(0.15, 0.3]     188405
Name: label, dtype: int64


## SF3079 — Data Validity Analysis

### Why does SF3079 show 99.9% drunk label?

**Accelerometer recording window:**
- Start: 1493739196712 ms → 15:33 UTC
- End:   1493790825495 ms → 05:53 UTC (~14 hours)

**TAC readings around accelerometer start:**

| TAC row | timestamp (ms)  | TAC value | label |
|---------|-----------------|-----------|-------|
| row 11  | 1493733728000   | 0.123     | 1     |
| row 12  | 1493735575000   | 0.181     | 1     |
| row 13  | 1493737421000   | 0.211     | 1     |

The accelerometer recording started **after the participant had already
been drinking** — TAC was already at 0.12~0.18 when data collection began.

**Two intoxication episodes were observed:**
- Episode 1: rows 11~23 (14:28 ~ 15:53 UTC)
- Episode 2: rows 31~43 (17:46 ~ 22:03 UTC)

### Why is TAC-based drunk% different from accelerometer-based drunk%?

| Metric              | Value  | Reason                                      |
|---------------------|--------|---------------------------------------------|
| TAC raw drunk%      | 48.1%  | 57 rows at 30-min intervals                 |
| Accelerometer drunk%| 99.9%  | 40Hz continuous — more rows in drunk period |

**Conclusion:** SF3079 is **not a data error**.
The high drunk% reflects the actual recording condition —
the participant was intoxicated for most of the accelerometer session.
This participant contains almost no sober accelerometer samples,
making it unsuitable for balanced evaluation.
Therefore, SF3079 is excluded from the test set and used
only as supplementary training data.

# pipeline Starting

In [25]:
for pid, group in df.groupby('pid'):
    print(f"{pid}: {len(group)} rows")

BK7610: 1225727 rows
BU4707: 447423 rows
CC6740: 2374695 rows
DC6359: 591358 rows
DK3500: 1339622 rows
HV0618: 1876013 rows
JB3156: 1177749 rows
JR8022: 307526 rows
MC7070: 318600 rows
MJ8002: 631303 rows
PC6771: 2141701 rows
SA0297: 962901 rows
SF3079: 662949 rows


In [26]:
print(df_tac['pid'].unique())

<StringArray>
['BK7610', 'BU4707', 'CC6740', 'DC6359', 'DK3500', 'HV0618', 'JB3156',
 'JR8022', 'MC7070', 'MJ8002', 'PC6771', 'SA0297', 'SF3079']
Length: 13, dtype: str


In [27]:
print(df_tac.shape)

(715, 5)


In [28]:
combined_df.head()

,time,pid,x,y,z,label,TAC_Reading
0,1493735870653,BK7610,0.1261,-0.0078,-0.0243,0,0.052238
1,1493735870679,BK7610,0.1336,-0.0697,-0.0446,0,0.052238
2,1493735870703,BK7610,0.1443,-0.0474,-0.0447,0,0.052238
3,1493735870729,BK7610,0.1255,-0.0038,0.0111,0,0.052238
4,1493735870753,BK7610,0.1076,0.0032,0.0276,0,0.052238


In [29]:
# Create magnitude column
combined_df['mag'] = (combined_df['x']**2 + combined_df['y']**2 + combined_df['z']**2)**0.5



In [30]:
''' 기존 extract_window_features 함수를 이걸로 교체
def extract_window_features_fast(group, window_size=100, step=50):
    current_pid = group.name

    # pandas Series → numpy 배열로 변환 (핵심 속도 개선)
    mag = group['mag'].values
    x   = group['x'].values
    y   = group['y'].values
    z   = group['z'].values
    lbl = group['label'].values

    if len(mag) < window_size:
        return pd.DataFrame()

    starts = np.arange(0, len(mag) - window_size, step)
    rows = []

    for s in starts:
        e   = s + window_size
        m   = mag[s:e]
        x_w = x[s:e]
        y_w = y[s:e]
        z_w = z[s:e]

        rows.append({
            'pid'     : current_pid,
            'label'   : int(lbl[s:e].mean() >= 0.5),
            'mag_mean': m.mean(),
            'mag_std' : m.std(),
            'mag_max' : m.max(),
            'mag_var' : m.var(),
            'x_std'   : x_w.std(),
            'y_std'   : y_w.std(),
            'z_std'   : z_w.std(),
            'jerk_std': np.std(np.diff(m)),
            'sway'    : np.sqrt(x_w**2 + z_w**2).mean() /
                        (np.abs(y_w).mean() + 1e-6),
        })

    return pd.DataFrame(rows)


# 루프로 실행 (진행상황 출력)
import time

results = []
for pid, group in combined_df.groupby('pid'):
    t = time.time()
    group.name = pid
    result = extract_window_features_fast(group)
    results.append(result)
    print(f"  {pid}: {len(result):,}개 윈도우 ({time.time()-t:.1f}초)")

windowed_data = pd.concat(results, ignore_index=True)
print(f"\n총 {len(windowed_data):,}개 윈도우, {windowed_data['pid'].nunique()}명")
print(windowed_data.head())

'''

' 기존 extract_window_features 함수를 이걸로 교체\ndef extract_window_features_fast(group, window_size=100, step=50):\n    current_pid = group.name\n\n    # pandas Series → numpy 배열로 변환 (핵심 속도 개선)\n    mag = group[\'mag\'].values\n    x   = group[\'x\'].values\n    y   = group[\'y\'].values\n    z   = group[\'z\'].values\n    lbl = group[\'label\'].values\n\n    if len(mag) < window_size:\n        return pd.DataFrame()\n\n    starts = np.arange(0, len(mag) - window_size, step)\n    rows = []\n\n    for s in starts:\n        e   = s + window_size\n        m   = mag[s:e]\n        x_w = x[s:e]\n        y_w = y[s:e]\n        z_w = z[s:e]\n\n        rows.append({\n            \'pid\'     : current_pid,\n            \'label\'   : int(lbl[s:e].mean() >= 0.5),\n            \'mag_mean\': m.mean(),\n            \'mag_std\' : m.std(),\n            \'mag_max\' : m.max(),\n            \'mag_var\' : m.var(),\n            \'x_std\'   : x_w.std(),\n            \'y_std\'   : y_w.std(),\n            \'z_std\'   :

In [ ]:
def extract_window_features(group, window_size=100):
    # 'group' is the data for a single PID
    # window_size=100 assumes roughly 2 seconds if sampling is 50Hz
    
    current_pid = group.name 
    
    features = []
    
    # Ensure there is enough data for at least one window
    if len(group) < window_size:
        return pd.DataFrame()

    for i in range(0, len(group) - window_size, window_size // 2): 
        window = group.iloc[i:i+window_size]
        
        feat = {
            'pid': current_pid,  
            'label': window['label'].iloc[0], 
            'mag_mean': window['mag'].mean(),
            'mag_std': window['mag'].std(),
            'mag_max': window['mag'].max(),
            'mag_var': window['mag'].var(),
            'x_std': window['x'].std(),
            'y_std': window['y'].std(),
            'z_std': window['z'].std(),
            # Jerk: std of first-order finite difference of acceleration magnitude
            # Captures movement irregularity — higher under intoxication
            # Jerk: 가속도 크기의 1차 차분의 표준편차
            # 움직임의 불규칙성을 측정 — 음주 시 높아짐
            'jerk_std': np.std(np.diff(window['mag'].values)),

            # Sway index: ratio of lateral (x-z plane) to vertical (y-axis) acceleration
            # Quantifies body sway relative to gravitational axis
            # Sway index: 수평(x-z 평면) 가속도 / 수직(y축) 가속도 비율
            # 중력 방향 대비 신체 흔들림 정도를 정규화한 지표
            'sway'    : (np.sqrt(window['x']**2 + window['z']**2)).mean() /
                        (np.abs(window['y']).mean() + 1e-6),
        }
        features.append(feat)
        
    return pd.DataFrame(features)

# Apply to each participant separately to avoid mixing data

windowed_data = (
    combined_df
    .groupby('pid')
    .apply(extract_window_features, include_groups=False)
    .reset_index(drop=True)
)




print(f"Created {len(windowed_data)} windows across {windowed_data['pid'].nunique()} participants.")
print(windowed_data.head())

Created 281132 windows across 13 participants.
      pid  label  mag_mean   mag_std   mag_max   mag_var     x_std     y_std  \
0  BK7610      0  0.072515  0.051678  0.242754  0.002671  0.041507  0.030790   
1  BK7610      0  0.042192  0.039420  0.205022  0.001554  0.025541  0.024937   
2  BK7610      0  0.080803  0.063333  0.302581  0.004011  0.041169  0.051209   
3  BK7610      0  0.075737  0.056229  0.302581  0.003162  0.033942  0.049087   
4  BK7610      0  0.033361  0.017356  0.111921  0.000301  0.012130  0.019139   

      z_std  jerk_std      sway  
0  0.059589  0.021024  2.844956  
1  0.042858  0.020918  1.949388  
2  0.077374  0.042496  1.879733  
3  0.065868  0.040687  1.780444  
4  0.028466  0.017130  1.725650  


In [19]:
X = windowed_data.drop(['label', 'pid'], axis=1)
y = windowed_data['label']
groups = windowed_data['pid']



In [20]:
# This ensures StandardScaler is fit ONLY on the training groups in each fold.
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

gkf = GroupKFold(n_splits=5)

# 'cross_validate' allows us to see multiple metrics (Accuracy, F1, etc.)
metrics = ['accuracy', 'precision', 'recall', 'f1']
results = cross_validate(pipeline, X, y, groups=groups, cv=gkf, scoring=metrics)

print(f"Mean Accuracy: {np.mean(results['test_accuracy']):.3f}")
print(f"Mean F1-Score: {np.mean(results['test_f1']):.3f}")
print(f"Mean Recall (Sensitivity): {np.mean(results['test_recall']):.3f}")

KeyboardInterrupt: 

## I need more time to change underline. I worked for increasing F1 score, making better model. 
### I am tring to make trianing set which cares rate of Drunk during the measuring 'accer' datas. I can find it is diffrent by pid. For example one thing is almost 0. and athor 99% we have to care about this, when split the data. 
## under line, the code is before calculating the rate of drunk. I will change as a result

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import pandas as pd
import numpy as np

# ─── 데이터 준비 ──────────────────────────────────────────────
feature_cols = [c for c in windowed_data.columns
                if c not in ['pid', 'label']]

X = windowed_data[feature_cols]
y = windowed_data['label']

# ─── 배분 정의 ────────────────────────────────────────────────
pid_seed  = 'JB3156'
pid_val   = 'CC6740'
pid_train = ['SA0297', 'DC6359', 'MJ8002', 'BK7610', 'DK3500']
pid_test  = ['PC6771', 'HV0618', 'SF3079', 'BU4707', 'JR8022']
train_order = [pid_seed, pid_val] + pid_train



# ─── 비교할 모델 정의 ─────────────────────────────────────────
# ImbPipeline: SMOTE가 포함될 때 사용 (imblearn 제공)
# SMOTE는 train 데이터에만 적용되도록 pipeline 안에 넣음

models = {
    # 1. Weighted Logistic Regression
    'Logistic (weighted)': ImbPipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            class_weight='balanced',  # 음주 24% 불균형 자동 보정
            max_iter=1000,
            random_state=42
        ))
    ]),

    # 2. Logistic Regression + SMOTE (오버샘플링)
    'Logistic + SMOTE': ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),  # 소수 클래스 합성 샘플 생성
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),

    # 3. Random Forest (class_weight)
    'Random Forest (weighted)': ImbPipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(
            n_estimators=200,
            class_weight='balanced',
            random_state=42
        ))
    ]),

    # 4. Random Forest + SMOTE
    'Random Forest + SMOTE': ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('clf', RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ))
    ]),

    # 5. Gradient Boosting (불균형에 강함)
    'Gradient Boosting': ImbPipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(
            n_estimators=200,
            random_state=42
        ))
    ]),
}

# ─── Walk-forward validation ──────────────────────────────────
print("=" * 60)
print("Walk-forward validation")
print("=" * 60)

# 모델별 fold 결과 저장
all_results = {name: [] for name in models}

for i in range(1, len(train_order)):
    seen  = train_order[:i]
    next_ = train_order[i]

    train_df = windowed_data[windowed_data['pid'].isin(seen)]
    test_df  = windowed_data[windowed_data['pid'] == next_]

    X_train = train_df[feature_cols]
    y_train = train_df['label']
    X_test  = test_df[feature_cols]
    y_test  = test_df['label']

    print(f"\nFold {i}: train={[p[:4] for p in seen]} → test={next_}")
    print(f"  train: {len(X_train):,}개 | test: {len(X_test):,}개 | "
          f"음주 비율 train={y_train.mean()*100:.1f}% test={y_test.mean()*100:.1f}%")

    for name, pipe in models.items():
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        f1_drunk  = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
        f1_sober  = f1_score(y_test, y_pred, pos_label=0, zero_division=0)
        f1_macro  = f1_score(y_test, y_pred, average='macro', zero_division=0)

        all_results[name].append({
            'fold'     : i,
            'test_pid' : next_,
            'f1_drunk' : round(f1_drunk, 3),
            'f1_sober' : round(f1_sober, 3),
            'f1_macro' : round(f1_macro, 3),
        })
        print(f"  [{name}] f1_drunk={f1_drunk:.3f} | "
              f"f1_sober={f1_sober:.3f} | f1_macro={f1_macro:.3f}")

# ─── Walk-forward 결과 요약 ───────────────────────────────────
print("\n" + "=" * 60)
print("Walk-forward 평균 성능 요약")
print("=" * 60)

summary_rows = []
for name, folds in all_results.items():
    df_fold = pd.DataFrame(folds)
    summary_rows.append({
        'model'         : name,
        'f1_drunk (avg)': round(df_fold['f1_drunk'].mean(), 3),
        'f1_sober (avg)': round(df_fold['f1_sober'].mean(), 3),
        'f1_macro (avg)': round(df_fold['f1_macro'].mean(), 3),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('f1_drunk (avg)', ascending=False)
print(summary_df.to_string(index=False))

# ─── 최종 모델 선택 → 홀드아웃 테스트 ───────────────────────
# walk-forward에서 f1_drunk 가장 높은 모델 자동 선택
best_model_name = summary_df.iloc[0]['model']
best_pipe = models[best_model_name]
print(f"\n최고 모델: {best_model_name}")

# 7명 전체로 재학습
train_final = windowed_data[windowed_data['pid'].isin(train_order)]
best_pipe.fit(train_final[feature_cols], train_final['label'])

# 홀드아웃 5명 테스트
print("\n" + "=" * 60)
print(f"홀드아웃 테스트 — {best_model_name}")
print("=" * 60)

holdout_rows = []
for pid in pid_test:
    test_df = windowed_data[windowed_data['pid'] == pid]
    y_true  = test_df['label']
    y_pred  = best_pipe.predict(test_df[feature_cols])

    f1_drunk = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)

    holdout_rows.append({
        'pid'      : pid,
        'n_windows': len(test_df),
        'drunk_pct': round(y_true.mean() * 100, 1),
        'f1_drunk' : round(f1_drunk, 3),
        'f1_macro' : round(f1_macro, 3),
    })

    print(f"\n{pid} (음주 {holdout_rows[-1]['drunk_pct']}%)")
    print(classification_report(y_true, y_pred,
                                 target_names=['sober', 'drunk'],
                                 zero_division=0))

print("\n홀드아웃 요약:")
print(pd.DataFrame(holdout_rows).to_string(index=False))

Walk-forward validation

Fold 1: train=['JB31'] → test=CC6740
  train: 23,553개 | test: 47,492개 | 음주 비율 train=18.9% test=18.3%
  [Logistic (weighted)] f1_drunk=0.667 | f1_sober=0.887 | f1_macro=0.777
  [Logistic + SMOTE] f1_drunk=0.667 | f1_sober=0.887 | f1_macro=0.777
  [Random Forest (weighted)] f1_drunk=0.647 | f1_sober=0.900 | f1_macro=0.773
  [Random Forest + SMOTE] f1_drunk=0.660 | f1_sober=0.897 | f1_macro=0.779
  [Gradient Boosting] f1_drunk=0.051 | f1_sober=0.899 | f1_macro=0.475

Fold 2: train=['JB31', 'CC67'] → test=SA0297
  train: 71,045개 | test: 19,257개 | 음주 비율 train=18.5% test=11.4%
  [Logistic (weighted)] f1_drunk=0.242 | f1_sober=0.326 | f1_macro=0.284
  [Logistic + SMOTE] f1_drunk=0.242 | f1_sober=0.329 | f1_macro=0.285
  [Random Forest (weighted)] f1_drunk=0.416 | f1_sober=0.870 | f1_macro=0.643
  [Random Forest + SMOTE] f1_drunk=0.418 | f1_sober=0.869 | f1_macro=0.644
  [Gradient Boosting] f1_drunk=0.359 | f1_sober=0.914 | f1_macro=0.637

Fold 3: train=['JB31', 'CC67'

In [25]:
# tac_dict 다시 만들기 (앞에서 만든 게 있으면 이 부분 건너뛰어도 됨)
tac_dir = 'clean_tac/'
tac_dict = {}
for fname in os.listdir(tac_dir):
    if fname.endswith('.csv'):
        pid = fname.replace('_clean_TAC.csv', '')
        df_tac = pd.read_csv(os.path.join(tac_dir, fname))
        tac_dict[pid] = df_tac

print("로드된 pid:", sorted(tac_dict.keys()))
# 13명 전체 TAC 원본 확인
print("=" * 60)
print("전체 TAC 원본 통계")
print("=" * 60)

tac_summary = []
for pid in sorted(tac_dict.keys()):
    tac = tac_dict[pid]
    above_08 = (tac['TAC_Reading'] >= 0.08).mean() * 100
    tac_summary.append({
        'pid'        : pid,
        'n_rows'     : len(tac),
        'min'        : round(tac['TAC_Reading'].min(), 4),
        'max'        : round(tac['TAC_Reading'].max(), 4),
        'mean'       : round(tac['TAC_Reading'].mean(), 4),
        'pct_>=0.08' : round(above_08, 1),
    })

tac_df = pd.DataFrame(tac_summary).sort_values('pct_>=0.08', ascending=False)
print(tac_df.to_string(index=False))

로드된 pid: ['BK7610', 'BU4707', 'CC6740', 'DC6359', 'DK3500', 'HV0618', 'JB3156', 'JR8022', 'MC7070', 'MJ8002', 'PC6771', 'SA0297', 'SF3079']
전체 TAC 원본 통계
   pid  n_rows     min    max   mean  pct_>=0.08
SF3079      54 -0.0283 0.2114 0.0687        48.1
MC7070      56 -0.0124 0.1967 0.0630        41.1
JR8022      47 -0.0122 0.2133 0.0671        40.4
CC6740      56 -0.0030 0.2447 0.0660        30.4
MJ8002      59 -0.0029 0.2416 0.0506        27.1
DC6359      55 -0.0043 0.1534 0.0435        25.5
BK7610      57 -0.0027 0.1718 0.0413        22.8
BU4707      57 -0.0027 0.1718 0.0413        22.8
SA0297      58 -0.0213 0.1826 0.0431        19.0
HV0618      54 -0.0064 0.1936 0.0390        16.7
DK3500      51 -0.0034 0.1275 0.0244         9.8
JB3156      55 -0.0012 0.0964 0.0294         7.3
PC6771      56 -0.0094 0.0919 0.0239         7.1


In [26]:
# combined_df 기준으로도 한 번 더 확인 (merge 후 레이블이 제대로 붙었는지)
print("\n" + "=" * 60)
print("combined_df 기준 레이블 분포 (merge 후)")
print("=" * 60)

merged_summary = []
for pid, group in combined_df.groupby('pid'):
    merged_summary.append({
        'pid'        : pid,
        'n_rows'     : len(group),
        'label_1_pct': round(group['label'].mean() * 100, 1),
        'tac_min'    : round(group['TAC_Reading'].min(), 4),
        'tac_max'    : round(group['TAC_Reading'].max(), 4),
        'tac_mean'   : round(group['TAC_Reading'].mean(), 4),
    })

merged_df_summary = pd.DataFrame(merged_summary).sort_values(
    'label_1_pct', ascending=False)
print(merged_df_summary.to_string(index=False))


combined_df 기준 레이블 분포 (merge 후)
   pid  n_rows  label_1_pct  tac_min  tac_max  tac_mean
SF3079  662949         99.9   0.0390   0.2052    0.1277
JR8022  307526         99.5   0.0506   0.2068    0.1366
MC7070  318600         93.1  -0.0002   0.1654    0.1270
BK7610 1225727         59.0   0.0417   0.1718    0.0960
MJ8002  631303         42.2  -0.0021   0.2416    0.0861
BU4707  447423         34.8   0.0417   0.0996    0.0654
JB3156 1177749         18.9   0.0024   0.0964    0.0619
CC6740 2374695         18.3   0.0039   0.2447    0.0485
DC6359  591358         11.7   0.0057   0.1221    0.0595
SA0297  962901         11.4  -0.0213   0.1826    0.0240
HV0618 1876013          4.6  -0.0033   0.1936    0.0174
PC6771 2141701          3.1  -0.0044   0.0900    0.0159
DK3500 1339622          0.0   0.0030   0.0712    0.0166


In [27]:
person_stats = windowed_data.groupby('pid').agg(
    n_windows   = ('label', 'count'),
    drunk_pct   = ('label', 'mean'),
    n_drunk     = ('label', 'sum'),
).reset_index()

person_stats['drunk_pct'] = (person_stats['drunk_pct'] * 100).round(1)
person_stats = person_stats.sort_values('drunk_pct', ascending=False)
print(person_stats.to_string(index=False))


   pid  n_windows  drunk_pct  n_drunk
SF3079      13257       99.9    13249
JR8022       6149       99.5     6119
MC7070       6370       93.1     5931
BK7610      24513       59.0    14470
MJ8002      12625       42.2     5323
BU4707       8947       34.8     3110
JB3156      23553       18.9     4457
CC6740      47492       18.3     8687
DC6359      11826       11.7     1389
SA0297      19257       11.4     2199
HV0618      37519        4.6     1738
PC6771      42833        3.1     1307
DK3500      26791        0.0        0
